# 🧠 Reasoning Pruning: Interactive Exploration

This notebook demonstrates how to use the **Reasoning Pruning (`rp`)** composable toolset directly in Google Colab / Jupyter.

### Core Workflow:
1. **Generate Trace ($G$)**: Produce reasoning steps for a question.
2. **Find Skip ($D$)**: Identify the first redundant/skippable reasoning step.
3. **Visualize Diff**: Render interactive colored trace diffs in notebook cells.
4. **Recursive Rollout**: Roll out multi-depth pruning transitions ($x \to y$).
5. **Interactive UI**: Launch the Gradio visualizer.

In [ ]:
# 1. Setup & Imports
# In Google Colab, install with:
# !curl -LsSf https://astral.sh/uv/install.sh | sh
# !uv sync --extra dev

import os
import reasoning_pruning as rp
from IPython.display import HTML, display

# Set your API keys (OpenAI, Anthropic, DeepSeek, etc.)
# os.environ["OPENAI_API_KEY"] = "your-api-key"
print(f"Reasoning Pruning version: {rp.__version__}")

## 1. Generate & Segment a Reasoning Trace

In [ ]:
question = "Janet buys 3 packs of 12 eggs. She bakes 2 cakes using 4 eggs each. How many eggs does she have left?"

# Generate reasoning steps using model G
trace = rp.generate_trace(question, model="gpt-4o-mini")

print(f"Total steps generated: {len(trace.steps)}")
for i, step in enumerate(trace.steps):
    print(f"[{i}] {step}")

## 2. Identify the First Skippable Step with Decision Model $D$

In [ ]:
decision = rp.find_first_skip(trace, decision_model="gpt-4o-mini")

print(f"Can Skip: {decision.can_skip}")
if decision.can_skip:
    print(f"Skipping step [{decision.skip_start_idx}]: {decision.skipped_steps}")
    print(f"Next useful step: {decision.next_step}")
    print(f"Justification: {decision.reason}")

## 3. Visualize Pruning Diff Directly in the Notebook

In [ ]:
# Render self-contained HTML diff with green prefix, red strikethrough, blue target
display(HTML(rp.render_trace_diff(trace, decision, as_html=True)))

## 4. Multi-Depth Recursive Rollout

In [ ]:
rollout = rp.rollout_pruning(
    question=question,
    generator_model="gpt-4o-mini",
    decision_model="gpt-4o-mini",
    max_depth=3,
)

print(f"Original Step Count: {rollout.original_step_count}")
print(f"Final Step Count:    {rollout.final_step_count}")
print(f"Compression Ratio:   {rollout.compression_ratio*100:.1f}%")
print(f"Extracted PT pairs:  {len(rollout.transitions)}")

for i, ex in enumerate(rollout.transitions):
    print(f"\n--- Transition Pair {i+1} (Depth {ex.depth}) ---")
    print(f"x: {ex.input_x}")
    print(f"y: {ex.target_y}")

## 5. Launch the Interactive Gradio Viewer

In [ ]:
# In Google Colab, share=True creates a public live link
# rp.launch_viewer(share=True)